Executive Summary & Model Selection Strategy
In handling class imbalance within multi-class datasets, balancing techniques can significantly impact a model's operational reliability. In this notebook, we evaluate two primary strategies for addressing minority class representation in a Random Forest model: Class Weighting (class_weight='balanced') and Random Oversampling (RandomOverSampler).

While oversampling slightly improved recall on minority class 3, Class Weighting is chosen as the final approach because it preserves high precision across rare classes, avoids artificial data duplication, and prevents model overfitting on rare instances.

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("/content/glass.csv")

Target column : df['type']
Feature columns :df.drop(columns=['type'])


In [9]:
df


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,1
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,7
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,7
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,7
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,7


In [10]:
df.isna().sum()

,0
RI,0
Na,0
Mg,0
Al,0
Si,0
K,0
Ca,0
Ba,0
Fe,0
Type,0


In [11]:
(df["Fe"]>0).sum()

np.int64(70)

The target variable (type) shows a severe class imbalance

In [13]:
df["Type"].value_counts()

,count
Type,
2,76
1,70
7,29
3,17
5,13
6,9


In [14]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=["Type"]),df["Type"],test_size=0.2,random_state=42)

Model Strategy: Class Weighting vs. Random Over-sampling

In [15]:
from sklearn.ensemble import RandomForestClassifier


In [16]:
model=RandomForestClassifier(random_state=42,class_weight="balanced")
model.fit(x_train,y_train)


RandomForestClassifier(class_weight='balanced', random_state=42)

In [17]:
pred=model.predict(x_test)

In [18]:
from  sklearn.metrics import classification_report
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           1       0.79      1.00      0.88        11
           2       0.85      0.79      0.81        14
           3       1.00      0.67      0.80         3
           5       1.00      0.50      0.67         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0.94         8

    accuracy                           0.86        43
   macro avg       0.92      0.83      0.85        43
weighted avg       0.87      0.86      0.85        43



In [19]:
from imblearn.over_sampling import RandomOverSampler
ros=RandomOverSampler(random_state=42)
x_train_resampled,y_train_resampled=ros.fit_resample(x_train,y_train)

In [20]:
modelresampled=RandomForestClassifier(random_state=42)
modelresampled.fit(x_train_resampled,y_train_resampled)

RandomForestClassifier(random_state=42)

In [21]:
predresampled=modelresampled.predict(x_test)

In [22]:
print(classification_report(y_test,predresampled))

              precision    recall  f1-score   support

           1       0.75      0.82      0.78        11
           2       0.91      0.71      0.80        14
           3       0.75      1.00      0.86         3
           5       1.00      1.00      1.00         4
           6       1.00      1.00      1.00         3
           7       0.89      1.00      0.94         8

    accuracy                           0.86        43
   macro avg       0.88      0.92      0.90        43
weighted avg       0.87      0.86      0.86        43

